# PEFT - LoRA

https://huggingface.co/docs/peft/en/developer_guides/lora


**1. 다양한 초기화 전략 (Initialization)**

LoRA 가중치를 어떻게 시작하느냐에 따라 학습 속도와 성능이 달라진다.

* **기본값:** 가중치 A는 Kaiming-uniform, B는 0으로 초기화하여 처음에는 정등 변환(Identity transform) 상태로 시작한다.
* **PiSSA:** 주특이값(Principal singular values)을 사용하여 초기화하며, 일반 LoRA보다 수렴 속도가 빠르고 성능이 우수하다.
* **OLoRA:** QR 분해를 활용하여 베이스 모델의 가중치를 변환하며, 학습 안정성과 수렴 속도를 높인다.
* **EVA:** 입력 활성화 값에 대해 SVD를 수행하여 데이터 기반으로 초기화하며, 레이어별로 랭크(Rank)를 유연하게 할당한다.

**2. 양자화 모델 최적화 (LoftQ & DoRA)**

* **LoftQ:** 양자화된 모델을 미세 조정할 때 발생하는 오차를 최소화하도록 LoRA 가중치를 초기화하는 기술이다.
* **DoRA (Weight-Decomposed Low-Rank Adaptation):** 가중치 업데이트를 크기(Magnitude)와 방향(Direction)으로 분리하여 처리하며, 특히 낮은 랭크에서도 높은 성능을 보여준다.

**3. 효율적인 추론 및 학습 기법**

* **aLoRA (Activated LoRA):** 특정 토큰(invocation tokens)이 나타날 때만 어댑터를 활성화하는 방식이다. 기본 모델과 KV 캐시를 공유할 수 있어 추론 속도를 획기적으로 높일 수 있다.
* **Rank-stabilized LoRA (rsLoRA):** 랭크()의 제곱근에 비례하여 스케일링을 조절함으로써, 높은 랭크를 사용할 때 학습을 안정화시킨다.
* **레이어 복제 (Layer Replication):** 기존 모델의 레이어를 논리적으로 복제하여 모델 크기를 확장하되, 실제 메모리 사용량은 최소화하면서 어댑터를 추가하는 방식이다.

**4. 고급 최적화 및 제어**

* **특수 옵티마이저:** 가중치 A를 고정하고 B만 튜닝하여 메모리를 아끼는 **LoRA-FA**, A와 B에 서로 다른 학습률을 적용해 속도를 2배 높이는 **LoRA+** 등을 지원한다.
* **세부 제어:** 특정 레이어마다 서로 다른 랭크()나 알파() 값을 지정할 수 있는 기능을 제공한다.
* **토큰 학습:** 특정 레이어의 가중치뿐만 아니라, 특정 토큰 임베딩만 선택적으로 학습시키는 기능을 지원한다.

In [8]:
%pip install -Uqqq transformers datasets accelerate trl peft hf_transfer pydantic langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [2]:
!nvidia-smi # GPU 확인 (모델명 / GPU ID / 메모리 사용량)

'nvidia-smi'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [3]:
# 로컬 기준 환경변수 설정
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

In [4]:
# Runpod 기준 환경변수 설정(Pod 에 환경 변수가 있어야 함)
# import os

# OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
# HF_TOKEN = os.environ['HF_TOKEN'] 

## 데이터셋 로드
https://huggingface.co/datasets/capybaraOh/naver-economy-news2stock

In [5]:
from datasets import load_dataset # HuggingFace 데이터셋 로더

# HuggingFace Hub train split 로드
dataset = load_dataset('capybaraOh/naver-economy-news2stock', split='train')
print(len(dataset))
dataset # Dataset 객체 정보

README.md:   0%|          | 0.00/347 [00:00<?, ?B/s]

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\datasets--capybaraOh--naver-economy-news2stock. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.27MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

1000


Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 1000
})

In [6]:
dataset[0] # {'system' : ... , 'user' : ... , 'assistatnt':...}

{'system': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n",
 'user': '추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대

In [10]:
# HuggingFace Dataset을 학습 / 평가 분리 후 Chat 메시지 포맷으로 변환
test_ratio = 0.2 # 평가셋 비율

train_data = [] # 학습 데이터 리스트
test_data = []  # 평가 데이터 리스트

data_indices = list(range(len(dataset))) # 전체 인덱스
test_size = int(len(dataset) * test_ratio) # 평가셋 크기

test_data_indices = data_indices[:test_size] # 앞부분은 평가셋 (인덱스)
train_data_indices = data_indices[test_size:] # 나머지는 학습셋 (인덱스)

# OpenAI / Chat 학습용 messages 포맷 : {'messages' : [{system},{user},{assistant}]}
def format_data(data):
    return {
        'messages' : [
            {
                'role' : 'system',
                'content' : data['system']
            },
            {
                'role' : 'user',
                'content' : data['user']
            },
            {
                'role' : 'assistant',
                'content' : data['assistant']
            },
        ]
    }

train_data = [format_data(dataset[i]) for i in train_data_indices] # 학습 인덱스 -> 학습 dict
test_data = [format_data(dataset[i]) for i in test_data_indices] # 평가 인덱스 -> 평가 dict

print(len(train_data))
print(len(test_data))



800
200


In [11]:
train_data[128] # message 포맷 dict 확인

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '에어부산 울란바토르·오사카 노선 재개\n에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는 일정으로 주 2회 운항한다.

In [12]:
# List -> HuggingFace Dataset (내용은 그대로, 컨테이너만 변경)
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

train_dataset[128]



{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '에어부산 울란바토르·오사카 노선 재개\n에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는 일정으로 주 2회 운항한다.

## NCSOFT/Llama-VARCO-8B-Instruct란?
https://huggingface.co/NCSOFT/Llama-VARCO-8B-Instruct


* **기반 모델:** Meta의 Llama-3.1-8B 모델을 기반으로 한다.
* **개발 목적:** 한국어 능력을 극대화하는 동시에 영어 구사 능력도 유지하도록 설계되었다.
* **학습 방법:** 한국어와 영어 데이터셋을 활용한 지속 사전 학습(Continual Pre-training)을 거쳤으며, 이후 지도 미세 조정(SFT)과 직접 선호도 최적화(DPO)를 통해 인간의 선호도에 맞게 정렬되었다.


**SFT에서 한국어능력향상과 동시에 영어능력유지란:**

일반적으로 한국어 데이터를 대량으로 추가 학습시키면 기존에 모델이 가지고 있던 영어 지식이 손상되는 '파괴적 망각(Catastrophic Forgetting)' 현상이 발생한다. 엔씨소프트는 이를 방지하기 위해 **지속 사전 학습(Continual Pre-training)**을 적용했다.

**_1. 데이터 믹스(Data Mixing) 전략:_**

단순히 한국어 데이터만 밀어 넣는 것이 아니라, 모델이 이미 학습했던 영어 데이터와 고품질의 한국어 데이터를 특정 비율로 섞어 학습한다. 이를 통해 기존의 영어 추론 능력을 '복습'하면서 새로운 언어 체계를 '습득'하게 된다.

**_2. 토크나이저 효율화와 임베딩 확장:_**

기존 Llama-3.1의 토크나이저 성능을 유지하면서 한국어 표현력을 높이기 위해 어휘 사전(Vocabulary)을 최적화한다. 영어 토큰 정보는 건드리지 않고 한국어 토큰의 밀도를 높여 두 언어 간의 연결 고리를 강화하는 방식이다.

**_3. 지식 전이(Knowledge Transfer):_**

영어 데이터로 학습된 모델의 강력한 논리적 사고 능력을 한국어로 전이시키는 과정을 거친다.

* **추론 능력 유지:** 수학이나 코딩 같은 논리적 작업은 영어 데이터에서 배운 구조를 그대로 활용한다.
* **언어 정렬:** SFT(지도 미세 조정) 단계에서 동일한 질문을 한국어와 영어로 번급하며 학습시켜, 언어에 상관없이 일관된 답변을 내놓도록 유도한다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM # 토크나이저 / 생성형 모델 자동 로더
import torch

pretrained_model_name = 'NCSOFT/Llama-VARCO-8B-Instruct' # 사전 학습 모델명

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name,
    dtype = torch.bfloat16, # 가중치 로딩 dtype(bf16)
    device_map= 'auto'      # 환경에 맞춰 CPU / GPU 자동 배치
)

# 해당 모델의 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\models--NCSOFT--Llama-VARCO-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## llama-3 chat template 변환

Llama3 모델은 특정 chat template 형식으로 학습되어, 그 형식을 사용해야 최적 성능을 낼 수 있다.
Chat template을 사용하지 않으면 모델이 대화 구조를 제대로 인식하지 못할 수 있다.
opean_ai 형식의 데이터를 llama-3 형식으로 변환한다.


**LLaMA-3 채팅 포맷**
LLaMA-3 채팅 포맷은 LLaMA-3 계열 챗봇 모델이 대화 내용을 이해하고 답변할 수 있도록 만들어진 입력 데이터 구조입니다.
여러 역할(시스템, 유저, 어시스턴트)의 메시지를 특별한 토큰과 구조로 묶어서 하나의 프롬프트로 합치는 방식입니다.
구조 예시
아래와 같이 대화 흐름을 명확히 구분하는 토큰들이 사용됩니다:

```
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
[시스템 역할 지침]<|eot_id|>
<|start_header_id|>user<|end_header_id|>
[유저 질문]<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
[모델의 답변]<|eot_id|>
```
* <|begin_of_text|> : 전체 프롬프트의 시작을 알리는 토큰
* <|start_header_id|>role<|end_header_id|> : 각 메시지의 역할 구분(시스템, 유저, 어시스턴트 등)
* 각 메시지 끝에 <|eot_id|> : 하나의 메시지 블록이 끝났음을 알림
* 마지막 assistant 블럭은 응답 생성 위치를 가리킨다. apply_chat_template(add_generation_prompt=False)로 설정했더라도 내부 템플릿에는 응답을 받을 자리 표시자로 <|assistant|> 토큰이 남아 있어, "여기서부터 어시스턴트가 답변을 생성해야 한다"는 신호를 제공하는 것임.

**왜 이 포맷이 필요할까?**

* 모델이 **“어디까지가 시스템 안내, 어디서부터가 유저 질문, 어디서부터가 답변인지”** 정확하게 파악할 수 있다.
* 여러 턴(turn)의 대화가 이어질 때도 메시지 경계를 명확히 구분해 혼동 없이 맥락을 유지할 수 있다.
* LLaMA-3 계열 모델은 이런 포맷으로 학습되어 있기 때문에 **실전 파인튜닝/추론 시에도 반드시 이 구조로 입력해야** 기대하는 챗봇 성능을 발휘할 수 있다.

In [2]:
# 하나의 샘플만 openai 방식 메시지 -> llama 3 방식 메시지로 변환
text = tokenizer.apply_chat_template(train_dataset[128]['messages'], tokenize=False)
print(text)

NameError: name 'tokenizer' is not defined

### data_collator 함수

* 미니배치(batch) 데이터를 모델이 바로 학습할 수 있는 형태(토큰·마스크·정답)로 변환합니다.
* 특히 아래와 같은 LLaMA-3 채팅 포맷을 쓸 때,
  “어디까지가 질문/어디서부터가 답변(assistant)인지”를 정확히 구분해서
  모델이 정답(답변 부분)만 학습하도록 레이블을 지정합니다.

#### 함수 설명

**1. 프롬프트 생성 (Prompt Construction)**

입력받은 `batch` 데이터는 리스트 내에 여러 메시지(`system`, `user`, `assistant`)를 포함하는 사전(dict) 구조이다.

* Llama 3의 특수 토큰(` <|begin_of_text|>`, `<|start_header_id|>`, `<|eot_id|>`)을 사용하여 모든 대화 내용을 하나의 긴 문자열로 병합한다.
* 각 역할(role)의 시작과 끝을 명확히 구분하여 모델이 대화 맥락을 이해할 수 있도록 구성한다.

**2. 토크나이즈 및 패딩 (Tokenization)**

병합된 문자열 리스트를 `tokenizer`를 통해 숫자 ID(`input_ids`)로 변환한다.

* `padding=True`: 배치 내의 문장들 중 가장 긴 문장을 기준으로 길이를 맞춘다.
* `truncation=True`: `max_length`를 초과하는 데이터는 절단한다.
* `return_tensors="pt"`: PyTorch 텐서 형식으로 결과를 반환한다.

**3. 레이블 생성 및 Loss Masking**

이 함수의 핵심 부분이다. 모델이 '사용자의 질문'이 아닌 **'모델의 답변(assistant)'** 부분에 대해서만 학습하도록 설정한다.

* **-100 값의 의미**: PyTorch의 `CrossEntropyLoss`는 레이블 값이 `-100`인 경우 손실(Loss) 계산에서 제외한다. 이를 통해 모델은 질문 부분을 예측하려고 노력하지 않고, 답변 부분의 정확도에만 집중하게 된다.
* **구간 탐색**: `assistant_tokens`를 기점으로 답변이 시작되는 위치를 찾고, `<|eot_id|>` 토큰이 나오는 지점까지의 인덱스를 추출한다.
* **값 복사**: 해당 구간의 `labels`에만 실제 `input_ids` 값을 복사하여 넣는다.

In [ ]:
# Lllam3 계열 Chat 학습용 데이터 콜레이터 : 프롬프트 생성 -> 토크나이즈/패딩 -> assistant 구간만 라벨링
# - 배치(messages)를 입력받아, 모델 학습 텐서로 변환
def data_collator(batch, tokenizer=tokenizer, max_length=8192):
    # 1. 프롬프트 생성
    prompts = []  # 배치 프롬프트 문자열을 담을 리스트
    for example in batch:
        prompt = '<|begin_of_text|>'  # 프롬프트 시작 토큰
        for msg in example['messages']:  # 샘플 내에서 (system/user/assistant) 순회
            role = msg['role']
            content = msg['content'].strip()
            # role + content로 템플릿을 완성
            prompt += f"<|start_header_id|>{role}<|end_header_id|>\n{content}<|eot_id|>"
        prompts.append(prompt)
    # print(prompts)

    # 2. 토큰처리 / 패딩 / 텐서 변환
    
    # 프롬프트를 토큰화해서 텐서로 변환
    tokenized = tokenizer(
        prompt,
        truncation = True,         # 최대 길이 초과시 자름
        max_length = max_length,   # 최대 길이 설정
        padding = True,            # 최대 길이 미만시 패딩 처리
        return_tensors = "pt"      # Pytorch Tensor 반환
    )
    input_ids = tokenized['input_ids']  # 토큰 id 텐서
    attention_mask = tokenized['attention_mask']  # 패딩 마스크 텐서
    # print(tokenized)
    # print(len(tokenized['input_ids'][0]))
    # print(len(tokenized['input_ids'][1]))
    # print(len(tokenized['attention_mask'][0]), len(tokenized['attention_mask'][0].sum().item()))
    # print(len(tokenized['attention_mask'][0]), len(tokenized['attention_mask'][0].sum().item()))

    # 3. 라벨 생성
    labels = torch.full_like(input_ids, fill_value=-100)  # input_ids shape으로 -100 기본값. (손실 계산 제외)
    print(labels.shape)

    assistant_header = '<|start_header_id|>assistant<|end_header_id|>\n'
    assistant_token_id = tokenizer.encode(assistant_header, add_special_tokens=False)  # 헤더의 토큰 패턴
    eot_token = '<|eot_id|>'
    eot_token_id = tokenizer.encode(eot_token, add_special_tokens=False)  # 종료 토큰의 토큰 패턴
    print(assistant_token_id)
    print(eot_token_id)

    for i, ids in enumerate(input_ids):  # 각 샘플별로 순회
        ids_list = ids.tolist()  # 슬라이싱 사용하기 위해 list로 변경
        
        # assistant 답변 시작위치 찾음
        # - <|start_header_id|>assistant<|end_header_id|>\n 다음 인덱스부터 답변으로 수집
        start = None  # 답변 시작 인덱스 초기화
        for idx in range(len(ids_list) - len(assistant_token_id) + 1):  # 헤더 길이만큼 탐색
            # assistant header 패턴과 매칭시
            if ids_list[idx: idx + len(assistant_token_id)] == assistant_token_id:
                start = idx + len(assistant_token_id)  # 헤더 다음 토큰부터 라벨링 시작
                break  # 첫 번째 assistant 구간 후 해당 for문 탈출
        
        # 답변 끝 위치 찾음 : <|eot_id|> 전까지
        if start is not None:  # assistant 헤더를 찾은 경우
            end = None  # 답변 종료 인덱스 초기화
            for idx in range(start, len(ids_list) - len(eot_token_id) + 1):  # start ~ 종료 토큰 전
                # start부터 종료 패턴과 매칭시
                if ids_list[idx: idx + len(eot_token_id)] == eot_token_id:
                    end = idx + len(eot_token_id)  # eot까지 포함한 구간 설정
                    break  # 첫 번째 eot 구간 후 해당 for문 탈출
        print(f"{i}: {start} ~ {end}")
        labels[i, start:end] = input_ids[i, start:end]  # assistant 답변 부분만 정답 라벨로 복사

    return {
        'input_ids': input_ids,  # 모델 입력
        'attention_mask': attention_mask,  # 패딩 마스크
        'labels': labels  # 손실 계산용 라벨(assistant 답변 부분만 labels 활용)
    }

data_collator([train_dataset[0], train_dataset[1]])

NameError: name 'tokenizer' is not defined

### Causal Language Model 파인튜닝: input_ids와 labels 구조 이해

**데이터 구조**
```
input_ids:  [system_tokens..., user_tokens..., assistant_tokens...]  # 전체 시퀀스
labels:     [-100, -100, ..., -100, assistant_tokens...]          # assistant만 학습 대상
```

| 항목 | 내용 |
| --- | --- |
| **Input IDs** | 프롬프트 + 정답 (전체 시퀀스) |
| **Labels** | `-100` (프롬프트 구간) + 정답 토큰 (답변 구간) |
| **결과** | 모델은 입력을 다 보지만, 오직 답변을 맞히는 과정에서만 학습이 일어남 |




> **질문에 해당하는 input_ids에 이미 답이 포함되어 있다!**
>
> **"답이 이미 있는데 어떻게 학습하는가?"**
>
> 모델은 정답을 "보면서" 각 위치에서 올바른 다음 토큰을 예측하는 법을 배운다. 마치 학생이 모범답안을 보며 "이 상황에서는 이렇게 답해야 한다"를 학습하는 것과 같다. 이것이 현대 LLM 파인튜닝의 핵심 메커니즘이다!


**_1. 인과적 언어 모델링 (Causal Language Modeling):_**

LLM(Llama, GPT 등)은 **이전 토큰들을 보고 다음 토큰을 예측**하는 방식으로 학습한다. 따라서 학습 데이터에는 프롬프트와 정답이 모두 포함된 전체 문장이 들어가야 한다.

* **학습 원리:** 모델은 번째 토큰까지를 입력으로 받아 번째 토큰을 예측한다.
* **구조:** `input_ids`가 `[A, B, C, D]`라면, 모델은 내부적으로 `A`를 보고 `B`를, `A, B`를 보고 `C`를 예측하는 과정을 동시에 수행한다.

**_2. Teacher Forcing 기법:_**
```
Position:   [0, 1, 2, 3, 4, 5, 6, 7, 8]
input_ids:  [A, B, C, D, E, F, G, H, I]
labels:     [-100, -100, -100, -100, E, F, G, H, I]
```

학습 과정:
- Position 4: A,B,C,D를 보고 → E 예측
- Position 5: A,B,C,D,E를 보고 → F 예측  
- Position 6: A,B,C,D,E,F를 보고 → G 예측

**_3. Labels와 Loss 계산의 역할:_**

`input_ids`에 정답이 포함되어 있더라도, 모델이 모든 구간에 대해 학습(손실 계산)을 수행하는 것은 아니다. 이때 중요한 역할을 하는 것이 바로 코드에 작성된 **`labels`**이다.

* **-100의 의미:** PyTorch의 `CrossEntropyLoss`는 기본적으로 레이블 값이 `-100`인 위치를 무시(ignore)한다.
* **학습 차단:** 코드에서 프롬프트(User 질문 등) 구간의 레이블을 `-100`으로 설정했기 때문에, 모델이 프롬프트 내용을 예측하며 발생하는 오차는 학습에 반영되지 않는다.
* **학습 집중:** 오직 `assistant`의 답변 구간에 해당하는 `labels`만 실제 `input_ids` 값을 가지므로, 모델은 **"프롬프트가 주어졌을 때 정답을 생성하는 방법"**에 대해서만 가중치를 업데이트한다.


**학습 vs 추론의 차이**

**_학습 시:_**
```
input_ids: <system>당신은 금융분석가</system><user>뉴스내용</user><assistant>분석결과</assistant>
labels:    [-100, -100, ..., -100, 분석결과_토큰들]
```

**_추론 시:_**
```
input:  <system>당신은 금융분석가</system><user>뉴스내용</user><assistant>
output: 분석결과 (모델이 한 토큰씩 생성)
```

In [ ]:
# 테스트 데이터로 변환된 결과 확인
example = train_dataset[128]
batch = data_collator([example]) # 배치 1개로 콜레이터 적용
print(f"{batch['input_ids'].shape}")    # input_ids 텐서 shape
print(f"{batch['attention_mask'].shape}") # attention_mask 텐서 shape
print(f"{batch['labels'].shape}") # labels 텐서 shape

In [ ]:
print(batch['input_ids'][0].tolist()) # 0번째 샘플 input_ids 리스트
print(batch['attention_mask'][0].tolist()) # 0번째 샘플 attention_mask 리스트
print(batch['labels'][0].tolist()) # 0번째 샘플 labels 리스트 (답변 제외 -100)

In [ ]:
# labels 에서 -100 제거한 후, assistant 정답 구간만 디코딩
label_ids = [token_id for token_id in batch['labels'][0].tolist() if token_id != -100]
text = tokenizer.decode(label_ids)
text

In [ ]:
# input_ids 를 토큰/문자 단위로 디코딩해서 확인
text_tokens = [] # 토큰 id를 디코딩한 문자열을 담을 리스트
for i, token_id in enumerate(batch['input_ids'][0].tolist()): # 토큰 ID 순회
    decoded_str = tokenizer.decode([token_id]) # 토큰 1개를 문자열로 디코딩
    text_tokens.append(decoded_str) # 디코딩 결과 누적

In [ ]:
# 데이터프레임 시각화

import pandas as pd

df = pd.DataFrame({
    'token':text_tokens, # 토큰 (디코딩된 문자열)
    'input_ids' : batch['input_ids'][0].tolist(), # 입력 토큰 ID
    'attention_mask' : batch['attention_mask'][0].tolist(), # 패딩 여부(1/0)
    'labels':batch['labels'][0].tolist() # 정답 라벨 (마스킹: -100)
}).transpose()

pd.set_option('display.max_columns', None) # 컬럼 생략 없음
df


## PEFT Finetuning - LoRA

* LoRA는 **"Low-Rank Adapter(저랭크 어댑터)"**
* 거대한 대형언어모델(LLM)의 **전체 파라미터를 일일이 미세조정(파인튜닝)하지 않고**,
  **딱 필요한 핵심 부분만 저렴하게 빠르게 학습**하는 최신 파인튜닝.
* **"LLM의 성능은 그대로, 비용/시간/메모리/유지보수는 최소로"** 파인튜닝을 할 수 있게 해주는 AI 실무에서 가장 중요한 기법 중 하나이다.

**왜 LoRA가 등장했을까?**

* GPT, Llama, DeepSeek 같은 대형언어모델은 **파라미터(매개변수) 수가 수십억\~수조 개**나 된다.
* 이런 모델을 파인튜닝하려면 **막대한 GPU 메모리와 시간, 저장 공간**이 필요.
* 하지만, 실제로 특정 태스크에 맞게 모델을 조정할 때 **전체를 다 바꿀 필요가 없다.**
* 대부분의 정보는 기존 모델에 이미 들어있고,
  **특정 입력(질문)과 특정 출력(답변)의 관계만 살짝 조정**해주면 충분하다.

**LoRA의 원리**

* 기존 대형 모델의 핵심 연산(주로 "곱셈" 부분)에
  **작고 얇은 "보조 네트워크(어댑터 레이어)"**를 덧붙인다.
* 전체 모델은 거의 건드리지 않고,
  **이 어댑터 레이어의 파라미터만 새로 추가해서 학습**
* 학습이 끝나면,

  * 원본 모델은 그대로
  * 어댑터(작은 추가 파라미터)만 별도로 저장하면 끝!
* 추론할 땐 **원본 모델 + LoRA 어댑터**를 합쳐서 쓸 수 있다.

**LoRA의 장점**

* **파인튜닝 비용(시간, 메모리, 저장 용량)이 압도적으로 절약**된다.
* 7B, 13B, 70B 등 대형 모델도
  **일반 GPU(24GB/48GB)로도 쉽게 파인튜닝**이 가능하다.
* **동일한 원본 모델에 다양한 LoRA 어댑터만 바꿔 끼우며
  다양한 분야별 파인튜닝 결과를 쉽게 쓸 수 있다.**

**LoRA와 기존 방식의 비교**

* **기존 파인튜닝:**
  전체 파라미터(수십\~수백 GB)를 새로 저장/관리/학습 → 비효율적
* **LoRA:**
  원본은 그대로 두고,
  변화가 필요한 부분(수 MB\~수십 MB)만 별도로 학습/저장


**실전에서의 활용 예시**

* 번역 LoRA, 요약 LoRA, 감정분석 LoRA 등
  **하나의 원본 모델에 여러 용도별 어댑터를 저장/관리**할 수 있다.
* **A100 80GB, 3090, T4 등 다양한 GPU 환경에서도
  고성능 LLM 튜닝이 매우 쉽게 가능하다.**

In [ ]:
# LoRA 설정 적용 후 학습 가능한 파라미터(Trainable) 확인
from peft import LoraConfig, get_peft_model # LoRA 설정 / 적용 함수

lora_config = LoraConfig(
    r = 8,              # 저랭크 행렬 rank
    lora_alpha=32,      # LoRA 스케일 계수 (alpha/r)
    lora_dropout=0.1,
    bias = "none",      # bias 학습 제외
    target_modules=['q_proj', 'v_proj'], # LoRA 를 주입할 모듈
    task_type = "CAUSAL_LM" # 작업 유형: 생성형
)

model = get_peft_model(model, lora_config) # 기존 모델에 LoRA 어댑터 적용
model.print_trainable_parameters() # 학습 가능한 파라미터 수 (비율) 



In [ ]:
#SFT(지도 미세 조정) 학습 설정

In [ ]:
from trl import SFTConfig  # TRL SFT 학습 설정 클래스

hub_model_id = 'kkkfu/Llama-VARCO-8b-news2stock-analyzer'  # 학습 완료 후 업로드할 Hub 모델 ID

sft_config = SFTConfig(  # SFT 학습 하이퍼파라미터/저장/로그 설정
    output_dir="Llama-VARCO-8b-news2stock-analyzer", # 학습 완료된 모델과 체크포인트가 저장될 경로이다.
    num_train_epochs=3,                              # 전체 데이터셋을 반복 학습할 횟수(Epoch)이다.
    per_device_train_batch_size=2,                   # 각 GPU(장치)당 한 번에 처리할 데이터 샘플의 개수이다.
    gradient_accumulation_steps=2,                   # 그래디언트를 2번 누적한 후 가중치를 업데이트한다. (실제 배치 크기 = 2 * 2 = 4 효과를 낸다.)
    gradient_checkpointing=True,                     # VRAM 절약을 위해 중간 활성화 값을 저장하지 않고 역전파 시 재계산하는 설정이다.
    optim="adamw_torch_fused",                       # 최적화 알고리즘 설정이다. fused 버전은 CUDA에서 더 빠르다.
    logging_steps=10,                                # 10 스텝마다 학습 로그(Loss 등)를 출력한다.
    save_strategy="steps",                           # 체크포인트 저장 기준을 'steps'(스텝 수)로 설정한다. (옵션: 'epoch')
    save_steps=50,                                   # 50 스텝마다 모델 체크포인트를 저장한다.
    bf16=True,                                       # BF16(Brain Float 16) 정밀도를 사용하여 메모리를 아끼고 연산 속도를 높인다. (Ampere GPU 이상 권장)
    learning_rate=1e-4,                              # 학습률(Learning Rate)이다. 가중치 업데이트의 크기를 결정한다.
    max_grad_norm=0.3,                               # 그래디언트 클리핑 임계값이다. 그래디언트 폭주를 막아 학습 안정성을 높인다.
    warmup_steps=0.03,                               # 전체 학습 단계의 3% 동안 학습률을 서서히 올리는 웜업(Warmup)을 수행한다.
    lr_scheduler_type="constant_with_warmup",                    # 학습률 스케줄러 타입이다. 여기서는 학습률을 변동 없이 상수로 유지한다.
    push_to_hub=True,                                # 학습이 끝나면 Hugging Face Hub에 모델을 자동으로 업로드한다.
    hub_model_id=hub_model_id,                       # Hub에 업로드될 때 사용될 저장소(Repository) ID이다.
    hub_token=True,                                  # Hub 업로드를 위해 인증 토큰을 사용한다.
    remove_unused_columns=False,                     # 데이터셋에서 모델의 forward 메서드 시그니처에 없는 컬럼을 자동으로 삭제하지 않도록 한다.
    dataset_kwargs={"skip_prepare_dataset": True},   # 데이터셋 처리 과정(packing 등)을 건너뛰도록 하는 설정이다.
    report_to=[],                                    # 학습 기록을 전송할 툴(WandB, Tensorboard 등)을 지정한다. 빈 리스트는 기록하지 않음을 의미한다.
    label_names=["labels"],                          # 손실(Loss) 계산 시 정답(Target)으로 사용할 데이터셋의 컬럼 이름이다.
    loss_type="nll"                                  # chunked_nll 비활성화 한다
)

In [ ]:
from trl import SFTTrainer # SFT 학습용 Trainer

trainer = SFTTrainer(
    model = model,      # LoRA 적용된 학습 모델
    args = sft_config,  # SFT 학습 설정
    train_dataset = train_dataset, # 학습 데이터 셋
    data_collator = data_collator  # 배치 텐서 생성 함수 
)

trainer.train()

# 평가

In [ ]:
# 테스트셋 messages에서 프롬프트/정답(assistant) 텍스트 분리
prompt_list = [] # 프롬프트 (assistant 답변 내용 이전) 리스트
label_list = [] # 정답 (assistant 답변 내용) 리스트

for messages in test_dataset["messages"]:
    # 채팅 템플릿 문자열로 반환
    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt =False)
    # assistant 답변 내용 전까지 input
    input = text.split('<|start_header_id|>user<|end_header_id|>\n')[0] + \
        '<|start_header_id|>user<|end_header_id|>\n'
    # assistant 답변 내용(종료 토큰 전) 추출
    labels = text.split('<|start_header_id|>user<|end_header_id|>\n')[1].split('<|eot.id|>')[0]
    prompt_list.append(input)
    label_list.append(labels)
    

In [ ]:
prompt_list[100]

In [ ]:
label_list[100]

### 추론모델 - 런타임결합
1. lora모델을 로드
2. base모델 + lora adapter 세팅

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline # 토크나이저 자동 로더 / 파이프라인 생성
import torch

peft_model_name = 'NCSOFT/Llama-VARCO-8B-Instruct' # 업로드 된 PEFT 모델 repo

finetuned_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name,
    dtype = torch.bfloat16, # 가중치 로딩 dtype(bf16)
    device_map= 'auto'      # 환경에 맞춰 CPU / GPU 자동 배치
)

# 해당 모델의 토크나이저 로드
# 같은 repo 에서 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(peft_model_name)

# 텍스트 생성 파이프라인
pipe = pipeline('text-generation', model=finetuned_model,tokenizer=tokenizer)
pipe

In [ ]:
# <|eot_id|>의 토큰 id 추출
eos_token = tokenizer('<|eot_id|>', add_special_tokens=False)['input_ids'][0]
eos_token

In [ ]:
# 테스트 추론 함수 : 프롬프트, 정답, 모델응답 비교 출력

def test_inference(pipe, prompt):
    # 파이프라인으로 결정론적인 답변 생성
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt) # 입력 구간 이후 생성 결과만 사용
    # 프롬프트 이후(생성된 부분)만 반환
    return outputs[0]['generated_text'][assistant_start:].strip() 

for prompt, label in zip(prompt_list[10:13], label_list[10:13]): # 10~12 샘플
    print(f"[prompt]: {prompt}")
    print(f"[label] : {label}")
    print(f"[response]: {test_inference(pipe,prompt)}")
    print('='*100)

# 추론

In [ ]:
# 뉴스 1건 입력받아 추론하는 함수

def inference(news) :
    messages = [
        {'role': 'system', 'content': '''
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''},  # 시스템 지시문
        {'role': 'user', 'content': news}
    ]
    
    # messages를 채팅 프롬프트 문자열로 반환
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt) # 입력 구간 이후 생성 결과만 사용
    # 프롬프트 이후(생성된 부분)만 반환
    return outputs[0]['generated_text'][assistant_start:].strip() 

In [ ]:
news = ''' 
현대건설-TS, 배송·주차로봇 다닐 아파트 안전 챙긴다

공동주택 내 미래모빌리티 대응 실증 MOU
단지 내 DRT·배송·서비스로봇 고려한 설계 연구
현대건설이 한국교통안전공단(TS)과 함께 공동주택 내 배송 로봇과 개인형 이동 수단 등 미래 모빌리티 도입 확대에 맞춘 안전기술을 개발하고 설계기준을 수립한다. 

현대건설은 지난 7일 웨스턴조선 서울에서 TS와 공동주택 미래 모빌리티 대응 안전기술 실증 협력을 위한 업무협약(MOU)을 체결했다고 8일 밝혔다. 이날 협약식에는 현대건설과 현대엔지니어링의 통합 R&D 조직인 HMG건설기술연구원의 김재영 원장과 송명준 인프라도시연구실장, 한국교통안전공단 민승기 모빌리티본부장 등이 참석했다.

현대건설에 따르면 이번 협약은 DRT(Demand Responsive Transport, 수요응답형교통)와 배송·서비스로봇, 퍼스널 모빌리티 등 공동주택에 다양한 이동 수단과 서비스 도입 확산을 계기로 이뤄졌다. 이에 맞춰 입주민의 보행 안전을 확보하고, 변화하는 교통 여건에 대응할 수 있는 단지 설계·운영기술을 개발하기 위해서라는 설명이다.

현대건설과 TS는 이번 협약을 통해 △공동주택 내 미래 모빌리티 안전성 증진 방안 공동 연구 △미래 모빌리티 도입에 따른 교통안전 개선사항 도출 △소프트웨어 중심 도로(Software Defined Road, SDR) 등의 기술을 활용한 도로환경 개선방안 검토 등을 추진한다. SDR은 소프트웨어로 도로 인프라를 제어·관리하고 다양한 이동 수단과 실시간으로 정보를 연계하는 차세대 도로 운영 기술이다.

양 기관은 DRT를 비롯해 배송·서비스로봇과 개인형 이동 수단 등 향후 주거단지에서 활용할 수 있는 모빌리티를 연구 대상에 포함해 안전한 이동환경을 구축한다는 계획이다. 단지 내 도로와 보행동선, 승하차·대기공간, 이동 수단별 운행체계 등 실제 주거환경을 구성하는 요소를 종합적으로 분석한 뒤 현장 실증을 통해 기술의 적용 가능성을 검증한다.

현대건설은 연구 결과를 토대로 입주민의 보행을 중심으로 단지 내 교통체계를 고도화한다. 모빌리티 운행 및 공간 이용 데이터를 활용해 도로와 동선 설계 최적화에 나선다는 것이다. 앞으로는 공동주택 기획·설계 단계부터 이동 수단별 운행 특성을 반영한 설계·운영 기준을 정립하고 미래 주거단지와 스마트시티를 연결하는 모빌리티 인프라 모델까지 발굴한다.

다양한 모빌리티 서비스가 주거단지 안으로 들어오면서 보행자와 이동 수단이 공존할 수 있는 공간계획과 운영체계가 중요하다는 게 현대건설의 설명이다. 특히 현대건설은 올해 압구정아파트지구 특별계획구역 재건축 사업 수주 과정에서 DRT를 비롯해 현대차그룹의 소형 모빌리티 '모베드(MobED)'를 비롯한 다양한 로봇과의 입주를 강조하기도 했다.▷관련기사: [압구정5 재건축]현대건설 승부수는 로봇, 그리고 갤러리아(2026년 5월21일)▷관련기사: "로봇과 함께 입주" 외친 현대건설의 '압구정 현대' 들여다보니…(2026년 5월12일)

현대건설 관계자는 "현대자동차·현대위아와 함께 공동주택 주차로봇 실증사업도 추진하고 있다"면서 "앞으로 주차로봇의 배치 및 운영기준을 정립하고 이를 단지 공간 계획과 연계한 스마트 주차 솔루션으로 발전시킬 것"이라고 말했다.
'''
inference(news)

In [ ]:
news = ''' 
국고채 금리, 유가 상승에 낙폭 되돌리며 상승 마감(종합)

(서울=연합뉴스) 강수지 기자 = 8일 국고채 금리는 장중 하락분을 반납하고 소폭 상승 마감했다.

외국인 국채선물 순매수에도 미국과 이란 간 교전이 재개되면서 국제유가가 배럴당 100달러에 육박한 영향을 받았다.

이날 서울 채권시장에서 3년 만기 국고채 금리는 전 거래일보다 0.1bp(1bp=0.01%포인트) 오른 연 3.901%에 장을 마쳤다.

10년물 금리는 연 4.401%로 1.6bp 상승했다. 5년물과 2년물은 각각 0.7bp, 0.1bp 상승해 연 4.127%, 연 3.721%에 마감했다.

20년물은 연 4.560%로 0.7bp 내렸다. 30년물과 50년물은 각각 0.4bp, 0.3bp 상승해 연 4.635%, 연 4.543%를 기록했다.

이날 3년 국채선물은 전일 대비 1틱 하락한 103.11에, 10년 국채선물은 4틱 하락한 105.13에 거래를 마쳤다. 외국인 순매수에 장중 강세폭을 확대했으나, 지정학적 위기로 인한 유가 우려에 상승분을 반납했다.

외국인은 3년 선물을 4천556계약, 10년 선물을 1천376계약 순매수했다.

간밤 미국 금융시장이 노동절로 휴장한 가운데 국내 채권시장은 외국인 국채선물 매수에 주목하며 장중 강세를 나타냈다.

그러나 유럽시장 개장 무렵 국제유가 벤치마크인 브렌트유가 배럴당 100달러에 육박하면서 국내 채권시장은 강세를 반납했다. 브렌트유 선물 가격은 이날 오후 1.4%가량 오른 배럴당 98.65달러 수준에서 거래되고 있다.

이날 발표된 한국의 2분기 실질 국내총생산(GDP) 성장률 잠정치는 전분기 대비 0.6%, 전년 대비 3.7%로 속보치와 동일했다. 명목 GDP는 전기 대비 9.2%, 전년 대비 26.4% 성장했다.

증권사의 한 채권 중개인은 "이날 채권은 강세를 보였는데, 미국과 이란 충돌 우려에 유가가 오르면서 강세를 되돌렸다"며 "유가와 미국 금리 움직임이 중요한 가운데 다음주 있을 국채선물 롤오버(월물 교체)에서 외국인 움직임이 중요할 듯하다"고 말했다.
'''
inference(news)

# Base 모델과 비교

In [ ]:
from transformers import AutoModelForCausalLM, pipeline

base_model_id = 'NCSOFT/Llama-VARCO-8B-Instruct' # 사전 학습 모델명

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype = torch.bfloat16, # 가중치 로딩 dtype(bf16)
    device_map= 'auto'      # 환경에 맞춰 CPU / GPU 자동 배치
)

base_pipe = pipeline('text-generation', model=finetuned_model,tokenizer=tokenizer)

# 베이스 모델과 LoRA 파인튜닝 모델의 응답과 정답 비교
for idx, (prompt, label) in enumerate(zip(prompt_list[10:13], label_list[10:13])):
    print(f"[샘플 {idx +1}")
    base_resp = test_inference(base_pipe, prompt)
    lora_resp = test_inference(pipe, prompt)
    print(f"[Base - 파인튜닝 전] {base_resp}")
    print(f"[LoRA - 파인튜닝 후] {lora_resp}")
    print(f"[Label] {label}")
    print("=" * 100)


In [ ]:
# HuggingFace에 finetuning 후 자동으로 업로드 되지 않은 경우 수동으로 업로드
import os  # 환경변수(HF_TOKEN) 사용
from huggingface_hub import HfApi  # Hub API 사용

repo_id = "kkkfu/Llama-VARCO-8b-news2stock-analyzer"  # 업로드할 모델 repo id
local_dir = "./Llama-VARCO-8b-news2stock-analyzer"  # 로컬 모델 폴더 경로

api = HfApi(token=os.environ["HF_TOKEN"])  # Hub 인증 토큰으로 API 객체 생성
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)  # repo가 없으면 생성(있으면 그대로 사용)

api.upload_folder(  # 로컬 폴더 전체를 Hub로 업로드
    folder_path=local_dir,  # 업로드할 로컬 폴더
    repo_id=repo_id,  # 대상 repo
    repo_type="model",  # 모델 repo로 업로드
    ignore_patterns=["checkpoint-*", "**/checkpoint-*"],  # 체크포인트 폴더는 제외
    commit_message="Upload final LoRA adapter (without checkpoints)"  # 커밋 메시지
)